# Fire Detection Model — Full Training (v2)

**Datasets:**
| Dataset | Classes | ~Images |
|---|---|---|
| maad-anwar-ertmq/smoke-and-fire-detection-a3dtx | fire, smoke | 1,844 |
| spyrobot/fire-smoke-and-human-detector | fire, smoke, civilian | 24,724 |
| juyeon-ko/firefighter | firefighter | ~500 |
| brainhack22/fallen-people (subsampled) | victim_down | 2,000 |
| freelance-projects/propane-cylinder | propane_tank | ~300 |
| wsfree/exit-cohy8 | exit_sign | ~400 |
| fire-extinguisher/fireextinguisher-z5atr | fire_extinguisher, exit_sign | ~800 |
| kauil/hazmat-h6fgb | hazmat_placard | 7,657 |

**Final classes:** `fire, smoke, firefighter, civilian, victim_down, propane_tank, exit_sign, fire_extinguisher, hazmat_placard`

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (free) or **A100** (Pro)
2. 🔑 Secrets tab → add `ROBOFLOW_KEY`

**Expected training time:** ~45-60 min on T4, ~20-25 min on A100

In [ ]:
# Cell 1 — Install dependencies
!pip install -q ultralytics roboflow python-dotenv

In [ ]:
# Cell 2 — Load API key
import os
from dotenv import load_dotenv

# Reads ROBOFLOW_KEY from .env in the same folder as this notebook
load_dotenv()
API_KEY = os.getenv("ROBOFLOW_KEY")
if not API_KEY:
    raise ValueError("ROBOFLOW_KEY not found — add it to your .env file")
print('API key loaded ✓')

In [ ]:
# Cell 3 — Dataset config
import random
from pathlib import Path

# (workspace, project, version, subsample_limit)
DATASETS = [
    ("maad-anwar-ertmq",   "smoke-and-fire-detection-a3dtx", 4,  None),
    ("spyrobot",           "fire-smoke-and-human-detector",  32, None),
    ("juyeon-ko",          "firefighter",                    1,  None),
    ("brainhack22",        "fallen-people---training-dataset-complete", 2, 2000),
    ("freelance-projects", "propane-cylinder",               1,  None),
    ("wsfree",             "exit-cohy8",                     2,  None),
    ("fire-extinguisher",  "fireextinguisher-z5atr",         2,  None),
    ("kauil",              "hazmat-h6fgb",                   16, None),
]

CLASS_REMAP = {
    'FIRE':  'fire',   'fire':  'fire',
    'SMOKE': 'smoke',  'smoke': 'smoke',
    'human': 'civilian',
    'firefighters': 'firefighter',  'firefighter': 'firefighter',
    'Fallen':   'victim_down',
    'Standing': None,
    'cylinder': 'propane_tank',
    'exit':      'exit_sign',
    'Fire_Exit': 'exit_sign',
    'Fire_Extinguisher': 'fire_extinguisher',
    'extinguisher':      'fire_extinguisher',
    'Flashing_Light_Orbs':      None,
    'Alarm_Activator':          None,
    'Sounders':                 None,
    'Fire_Blanket':             None,
    'Fire_Suppression_Signage': None,
    'non_flammable_gas':        'hazmat_placard',
    'blasting_agents':          'hazmat_placard',
    'flammable_solid':          'hazmat_placard',
    'spontaneously_combustible':'hazmat_placard',
    'oxygen':                   'hazmat_placard',
    'dangerous_when_wet':       'hazmat_placard',
    'explosives':               'hazmat_placard',
    'oxidizer':                 'hazmat_placard',
    'inhalation_hazard':        'hazmat_placard',
    'corrosive':                'hazmat_placard',
    'radioactive':              'hazmat_placard',
    'organic_peroxide':         'hazmat_placard',
    'fuel_oil':                 'hazmat_placard',
    'poison':                   'hazmat_placard',
    'flammable_gas':            'hazmat_placard',
}

CLASSES = [
    'fire', 'smoke', 'firefighter', 'civilian', 'victim_down',
    'propane_tank', 'exit_sign', 'fire_extinguisher', 'hazmat_placard',
]

MERGED = Path('datasets/merged')
RAW    = Path('datasets/raw')
print(f'{len(CLASSES)} classes: {CLASSES}')

In [ ]:
# Cell 4 — Download all datasets
from roboflow import Roboflow

rf = Roboflow(api_key=API_KEY)
download_paths = []

for ws, proj, ver, _ in DATASETS:
    print(f'Downloading {ws}/{proj} v{ver}...')
    ds = rf.workspace(ws).project(proj).version(ver).download(
        'yolov8', location=str(RAW / f'{proj}-v{ver}')
    )
    download_paths.append((Path(ds.location), _))  # path, subsample_limit
    print(f'  → {ds.location}')

print('\nAll datasets downloaded ✓')

In [ ]:
# Cell 5 — Merge into unified YOLO dataset
import shutil
import yaml
import random

for split in ('train', 'valid'):
    (MERGED / split / 'images').mkdir(parents=True, exist_ok=True)
    (MERGED / split / 'labels').mkdir(parents=True, exist_ok=True)

totals = {'train': 0, 'valid': 0}

for ds_path, subsample_limit in download_paths:
    yaml_path = ds_path / 'data.yaml'
    if not yaml_path.exists():
        print(f'  ⚠️  No data.yaml in {ds_path}')
        continue

    with open(yaml_path) as f:
        meta = yaml.safe_load(f)
    ds_classes = meta.get('names', [])

    for split in ('train', 'valid'):
        img_dir = ds_path / split / 'images'
        lbl_dir = ds_path / split / 'labels'
        if not img_dir.exists():
            continue

        candidates = list(img_dir.iterdir())

        # Subsample if limit set (apply proportionally across splits)
        if subsample_limit is not None:
            random.seed(42)
            candidates = random.sample(candidates, min(subsample_limit, len(candidates)))

        for img_file in candidates:
            if img_file.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
                continue
            lbl_file = lbl_dir / img_file.with_suffix('.txt').name
            if not lbl_file.exists():
                continue

            new_lines = []
            for line in lbl_file.read_text().strip().splitlines():
                parts = line.split()
                if not parts:
                    continue
                old_idx = int(parts[0])
                if old_idx >= len(ds_classes):
                    continue
                canonical = CLASS_REMAP.get(ds_classes[old_idx])
                if canonical is None or canonical not in CLASSES:
                    continue
                new_lines.append(f"{CLASSES.index(canonical)} " + " ".join(parts[1:]))

            if not new_lines:
                continue

            prefix = ds_path.name.replace(' ', '_')
            shutil.copy2(img_file,
                MERGED / split / 'images' / f'{prefix}_{img_file.name}')
            (MERGED / split / 'labels' / f'{prefix}_{lbl_file.name}').write_text('\n'.join(new_lines))
            totals[split] += 1

    print(f'Merged {ds_path.name}')

# Write data.yaml
with open(MERGED / 'data.yaml', 'w') as f:
    yaml.dump({
        'path':  str(MERGED),
        'train': 'train/images',
        'val':   'valid/images',
        'nc':    len(CLASSES),
        'names': CLASSES,
    }, f, default_flow_style=False)

print(f'\nMerge complete → train: {totals["train"]}  valid: {totals["valid"]} images ✓')

In [ ]:
# Cell 6 — Class distribution check
from collections import Counter

counts = Counter()
for lbl_file in (MERGED / 'train' / 'labels').iterdir():
    for line in lbl_file.read_text().strip().splitlines():
        parts = line.split()
        if parts:
            counts[CLASSES[int(parts[0])]] += 1

print('Training label distribution:')
for cls, count in sorted(counts.items(), key=lambda x: -x[1]):
    bar = '█' * (count // 200)
    print(f'  {cls:20s} {count:6d}  {bar}')

In [ ]:
# Cell 7 — Train
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=str(MERGED / 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,           # GPU 0 (RTX 5070 Ti)
    project='runs',
    name='seventhousand',
    exist_ok=True,
)

In [ ]:
# Cell 8 — Evaluate
best = YOLO('runs/seventhousand/weights/best.pt')
metrics = best.val(data=str(MERGED / 'data.yaml'), device=0)
print(f'mAP50:    {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')
print('Per-class mAP50:')
for cls, ap in zip(CLASSES, metrics.box.ap50):
    print(f'  {cls:20s} {ap:.3f}')

In [ ]:
# Cell 9 — Weights location
import os
weights = os.path.abspath('runs/seventhousand/weights/best.pt')
print(f'Weights saved to: {weights}')